In [67]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

from nba_crunch.data_processing import enrich_free_throws, season_from_game_id

pd.set_option('display.max_columns', None)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
def test_player(row):
    """
    Run a two-proportion z-test for one player's crunch vs non-crunch FT%.
    
    Parameters
    ----------
    row : pd.Series
        A row from the qualified player dataframe with columns:
        crunch_makes, crunch_attempts, non_crunch_makes, non_crunch_attempts.
    
    Returns
    -------
    pd.Series
        With keys: z_stat, p_value, ci_low, ci_high
    """

    crunch_makes = row['crunch_makes']
    crunch_attempts = row['crunch_attempts']
    non_crunch_makes = row['non_crunch_makes']
    non_crunch_attempts = row['non_crunch_attempts']

    counts = np.array([crunch_makes, non_crunch_makes])
    totals = np.array([crunch_attempts, non_crunch_attempts])
    z_stat, p_value = proportions_ztest(counts, totals)

    ci_low, ci_high = confint_proportions_2indep(
            count1=crunch_makes,
            nobs1=crunch_attempts,
            count2=non_crunch_makes,
            nobs2=non_crunch_attempts,
            method='wald',
            alpha=0.05
        )

    return pd.Series({
        'z_stat': z_stat,
        'p_value': p_value,
        'ci_low': ci_low,
        'ci_high': ci_high
    })

In [2]:
pbp_dir = Path('../data/raw/pbp')
all_files = list(pbp_dir.glob('*.csv'))
print(f"Found {len(all_files)} files")

dfs = []
errors = []

for filepath in tqdm(all_files):
    try:
        df = pd.read_csv(filepath, dtype={'gameId': str, 'scoreHome': str, 'scoreAway': str})
        df_enriched = enrich_free_throws(df)
        df_enriched['season'] = season_from_game_id(df_enriched['gameId'].iloc[0])
        dfs.append(df_enriched)
    except Exception as e:
        errors.append((filepath.name, str(e)))

all_fts = pd.concat(dfs, ignore_index=True)
print(f"Total FTs: {len(all_fts)}")
print(f"Errors: {len(errors)}")

Found 3690 files


100%|██████████| 3690/3690 [00:10<00:00, 346.72it/s]


Total FTs: 164510
Errors: 0


In [3]:
all_fts.info()

<class 'pandas.DataFrame'>
RangeIndex: 164510 entries, 0 to 164509
Data columns (total 29 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   gameId          164510 non-null  str    
 1   actionNumber    164510 non-null  int64  
 2   clock           164510 non-null  str    
 3   period          164510 non-null  int64  
 4   teamId          164510 non-null  int64  
 5   teamTricode     164510 non-null  str    
 6   personId        164510 non-null  int64  
 7   playerName      164510 non-null  str    
 8   playerNameI     164510 non-null  str    
 9   xLegacy         164510 non-null  int64  
 10  yLegacy         164510 non-null  int64  
 11  shotDistance    164510 non-null  int64  
 12  shotResult      0 non-null       str    
 13  isFieldGoal     164510 non-null  int64  
 14  scoreHome       164510 non-null  int64  
 15  scoreAway       164510 non-null  int64  
 16  pointsTotal     164510 non-null  int64  
 17  location        16451

In [12]:
crunch_fts = all_fts[all_fts['is_crunch'] == True]
non_crunch_fts = all_fts[all_fts['is_crunch'] == False]

crunch_attempts_per_player = crunch_fts.groupby('personId').size()
print(crunch_attempts_per_player.describe())
print(crunch_attempts_per_player.quantile([0.5, 0.75, 0.9, 0.95]))

count    455.000000
mean      20.663736
std       26.071300
min        1.000000
25%        4.000000
50%       10.000000
75%       27.000000
max      161.000000
dtype: float64
0.50    10.0
0.75    27.0
0.90    53.8
0.95    73.0
dtype: float64


In [17]:
crunch_fts.groupby(['personId', 'playerName']).size().sort_values(ascending=False)

personId  playerName        
203999    Jokić                 161
201942    DeRozan               143
1629027   Young                 135
1628983   Gilgeous-Alexander    131
1630178   Maxey                 128
                               ... 
202709    Joseph                  1
1629004   Mykhailiuk              1
203914    Harris                  1
1631204   Sasser                  1
1641931   Bitim                   1
Length: 455, dtype: int64

In [18]:
# Compute SE of a proportion for a few reference n values, at p = 0.75
# Formula: SE = sqrt(p * (1-p) / n)

p = 0.75
sample_sizes = [10, 30, 50, 100, 500]

for n in sample_sizes:
    se = np.sqrt(p * (1 - p) / n)
    print(f"n = {n:4d}: SE = {se:.4f} ({se*100:.1f} percentage points)")

n =   10: SE = 0.1369 (13.7 percentage points)
n =   30: SE = 0.0791 (7.9 percentage points)
n =   50: SE = 0.0612 (6.1 percentage points)
n =  100: SE = 0.0433 (4.3 percentage points)
n =  500: SE = 0.0194 (1.9 percentage points)


In [60]:
crunch_agg = (
    crunch_fts.groupby('personId')
    .agg(
        playerName=('playerNameI', 'first'),
        crunch_makes=('made', 'sum'),
        crunch_attempts=('made', 'count'),
    )
)

crunch_agg['crunch_pct'] = crunch_agg['crunch_makes'] / crunch_agg['crunch_attempts']

non_crunch_agg = (
    non_crunch_fts.groupby('personId')
    .agg(
        non_crunch_makes=('made', 'sum'),
        non_crunch_attempts=('made', 'count'),
    )
)

non_crunch_agg['non_crunch_pct'] = non_crunch_agg['non_crunch_makes'] / non_crunch_agg['non_crunch_attempts']

player_stats = crunch_agg.merge(non_crunch_agg, on='personId', how='inner')
player_stats['diff'] = player_stats['crunch_pct'] - player_stats['non_crunch_pct']

In [63]:
qualified = player_stats[player_stats['crunch_attempts'] >= 50].copy()
qualified.sort_values(by='diff').head(10)

,playerName,crunch_makes,crunch_attempts,crunch_pct,non_crunch_makes,non_crunch_attempts,non_crunch_pct,diff
personId,,,,,,,,
1628369,J. Tatum,45,64,0.703125,792,952,0.831933,-0.128808
1641705,V. Wembanyama,67,93,0.720430,755,913,0.826944,-0.106514
1627759,J. Brown,50,74,0.675676,833,1084,0.768450,-0.092775
1627783,P. Siakam,36,57,0.631579,742,1025,0.723902,-0.092323
1631101,S. Sharpe,44,61,0.721311,399,497,0.802817,-0.081505
1628389,B. Adebayo,38,55,0.690909,834,1083,0.770083,-0.079174
1627750,J. Murray,83,102,0.813725,638,718,0.888579,-0.074854
202710,J. Butler III,47,60,0.783333,925,1078,0.858071,-0.074737
1630578,A. Sengun,79,126,0.626984,721,1030,0.700000,-0.073016


In [65]:
qualified.sort_values(by='diff', ascending=False).head(10)

,playerName,crunch_makes,crunch_attempts,crunch_pct,non_crunch_makes,non_crunch_attempts,non_crunch_pct,diff
personId,,,,,,,,
1628970,M. Bridges,51,55,0.927273,552,663,0.832579,0.094694
1628976,W. Carter Jr.,47,59,0.796610,373,500,0.746000,0.050610
1626181,N. Powell,57,66,0.863636,562,689,0.815675,0.047961
203507,G. Antetokounmpo,75,110,0.681818,1107,1736,0.637673,0.044145
1630166,D. Avdija,64,78,0.820513,917,1177,0.779099,0.041413
1629627,Z. Williamson,41,56,0.732143,797,1144,0.696678,0.035465
1628991,J. Jackson Jr.,53,64,0.828125,758,954,0.794549,0.033576
201942,D. DeRozan,126,143,0.881119,1148,1341,0.856078,0.025041
1628368,D. Fox,61,77,0.792208,701,910,0.770330,0.021878


In [72]:
test_results = qualified.apply(test_player, axis=1)
qualified = pd.concat([qualified, test_results], axis=1)

In [77]:
qualified.sort_values(by='diff')


,playerName,crunch_makes,crunch_attempts,crunch_pct,non_crunch_makes,non_crunch_attempts,non_crunch_pct,diff,z_stat,p_value,ci_low,ci_high,p_adj,significant_fdr
personId,,,,,,,,,,,,,,
1628369,J. Tatum,45,64,0.703125,792,952,0.831933,-0.128808,-2.618232,0.008839,-0.243234,-0.014382,0.278446,False
1641705,V. Wembanyama,67,93,0.720430,755,913,0.826944,-0.106514,-2.531266,0.011365,-0.200968,-0.012060,0.278446,False
1627759,J. Brown,50,74,0.675676,833,1084,0.768450,-0.092775,-1.814541,0.069595,-0.202348,0.016799,0.613471,False
1627783,P. Siakam,36,57,0.631579,742,1025,0.723902,-0.092323,-1.509380,0.131202,-0.220506,0.035859,0.673656,False
1631101,S. Sharpe,44,61,0.721311,399,497,0.802817,-0.081505,-1.485238,0.137481,-0.199331,0.036320,0.673656,False
1628389,B. Adebayo,38,55,0.690909,834,1083,0.770083,-0.079174,-1.353475,0.175904,-0.203848,0.045500,0.718274,False
1627750,J. Murray,83,102,0.813725,638,718,0.888579,-0.074854,-2.171193,0.029917,-0.153837,0.004129,0.488638,False
202710,J. Butler III,47,60,0.783333,925,1078,0.858071,-0.074737,-1.596267,0.110429,-0.181040,0.031566,0.673656,False
1630578,A. Sengun,79,126,0.626984,721,1030,0.700000,-0.073016,-1.675831,0.093771,-0.161974,0.015942,0.656400,False


In [75]:
from statsmodels.stats.multitest import multipletests

# multipletests returns 4 things: (rejected, adjusted_p, alpha_sidak_corrected, alpha_bonferroni_corrected)
# We care about the first two.

reject, p_adj, _, _ = multipletests(qualified['p_value'], alpha=0.05, method='fdr_bh')

qualified['p_adj'] = p_adj
qualified['significant_fdr'] = reject

# Look at survivors
significant = qualified[qualified['significant_fdr']]
print(f"Players surviving FDR correction: {len(significant)}")
significant[['playerName', 'crunch_attempts', 'crunch_pct', 'non_crunch_pct', 'diff', 'p_value', 'p_adj']]

Players surviving FDR correction: 0


,playerName,crunch_attempts,crunch_pct,non_crunch_pct,diff,p_value,p_adj
personId,,,,,,,
